In [2]:
import os
import time
import uuid
import warnings
import mhcgnomes
import traceback
import numpy as np
import pandas as pd

In [3]:
PROTEIN_CHECK="^[ACDEFGHIKLMNPQRSTVWY]+$"
HOST_SPECIES = ['human', 'mouse']
CDR3_CHAINS = ['alpha', 'beta']

def fix_mhc_name(allele_str, chain = 'alpha'):
    try:
        # Предварительные проверки и подготовления
        if pd.isna(allele_str):
            return pd.NA     
        allele_first = allele_str.split(" ")[0]
        if allele_first == "B2M":
            return "B2M"
        # Получение конкретной аллели
        parsed_allele = mhcgnomes.parse(allele_first)
        if isinstance(parsed_allele, mhcgnomes.pair.Pair):
            if chain == 'alpha':
                allele = parsed_allele.alpha
            elif chain == 'beta':
                allele = parsed_allele.beta
            else:
                raise ValueError('Unknown chain')
        elif isinstance(parsed_allele, mhcgnomes.allele.Allele):
            allele = parsed_allele
        else:
            raise ValueError('Not allele')
        # Проверка
        if allele.gene.species.name == "Homo sapiens":
            if len(allele.allele_fields) < 2:
                raise ValueError('Too many allele fields. Need at least 2.')
            elif len(allele.allele_fields) == 2:
                return allele.to_string()
            else:
                return allele.restrict_allele_fields(2, drop_annotations=True, drop_mutations=True).to_string()
        elif allele.gene.species.name == "Mus musculus":
            return allele.to_string()
        else:
            raise ValueError('Wrong species. Only human and mouse are allowed.')
    except (mhcgnomes.errors.ParseError, TypeError, ValueError):
        return pd.NA

def calculate_receptor_id(data: pd.DataFrame, column: str) -> pd.DataFrame:
    cleaned_data = data.copy(deep=True)
    unique_id = cleaned_data[column].unique()
    replacement = {k:str(uuid.uuid4()) for k in unique_id}
    cleaned_data["id"] = cleaned_data[column].map(replacement)
    return cleaned_data

In [5]:
!pwd

/home/stotoshka/Documents/RNIMU/dissertation_new/TCRpred/notebooks/tests


In [10]:
raw_data = pd.read_csv("../../data/raw-data/VDJdb/VDJdb.csv",sep=";",header = 0)
raw_data.head()

,Gene,CDR3,V,J,Species,MHC A,MHC B,MHC class,Epitope,Epitope gene,Epitope species,Reference,Method,Meta,CDR3fix,Score,receptor_id
0,TRB,CASSIVGGNEQFF,TRBV19*01,TRBJ2-1*01,HomoSapiens,HLA-A*02:01,B2M,MHCI,GILGFVFTL,M,InfluenzaA,PMID:28629751,"{""identification"": ""tetramer-sort "", ""frequenc...","{""study.id"": """", ""cell.subset"": ""CD8+"", ""subje...","{""cdr3"": ""CASSIVGGNEQFF"", ""cdr3_old"": ""CASSIVG...",3,0
1,TRB,CASSMRSTGELFF,TRBV19*01,TRBJ2-2*01,HomoSapiens,HLA-A*02:01,B2M,MHCI,GILGFVFTL,M,InfluenzaA,PMID:28629751,"{""identification"": ""tetramer-sort "", ""frequenc...","{""study.id"": """", ""cell.subset"": ""CD8+"", ""subje...","{""cdr3"": ""CASSMRSTGELFF"", ""cdr3_old"": ""CASSMRS...",3,0
2,TRB,CASSIRSAWAQYF,TRBV19*01,TRBJ2-3*01,HomoSapiens,HLA-A*02:01,B2M,MHCI,GILGFVFTL,M,InfluenzaA,PMID:28629751,"{""identification"": ""tetramer-sort "", ""frequenc...","{""study.id"": """", ""cell.subset"": ""CD8+"", ""subje...","{""cdr3"": ""CASSIRSAWAQYF"", ""cdr3_old"": ""CASSIRS...",2,0
3,TRB,CASSQRSTGELFF,TRBV19*01,TRBJ2-2*01,HomoSapiens,HLA-A*02:01,B2M,MHCI,GILGFVFTL,M,InfluenzaA,PMID:28629751,"{""identification"": ""tetramer-sort "", ""frequenc...","{""study.id"": """", ""cell.subset"": ""CD8+"", ""subje...","{""cdr3"": ""CASSQRSTGELFF"", ""cdr3_old"": ""CASSQRS...",2,0
4,TRB,CASSIRSSYEQYF,TRBV19*01,TRBJ2-7*01,HomoSapiens,HLA-A*02:01,B2M,MHCI,GILGFVFTL,M,InfluenzaA,PMID:28629751,"{""identification"": ""tetramer-sort "", ""frequenc...","{""study.id"": """", ""cell.subset"": ""CD8+"", ""subje...","{""cdr3"": ""CASSIRSSYEQYF"", ""cdr3_old"": ""CASSIRS...",3,0


In [11]:
len(raw_data['receptor_id'].unique())

87053

In [14]:
raw_data = pd.read_csv("../../data/clean-data/VDJdb_clean.csv",sep=";",header = 0)
raw_data.head()

,id,J_alpha,J_beta,V_alpha,V_beta,cdr3_alpha,cdr3_beta,epitope,mhc_alpha,mhc_beta,mhc_class,host_species,epitope_species,epitope_source,database,D_alpha,D_beta
0,000009ea-df25-473e-be6e-292d78e17cc6,NaN,TRBJ1-6*01,NaN,TRBV11-2*01,NaN,CASSLAPPGRLNSPLHF,PKYVKQNTLKLAT,HLA-DRA*01:01,HLA-DRB1*04:01,II,human,InfluenzaA,HA,VDJdb,NaN,NaN
1,0000d05c-5e1d-428d-9182-4e620f6821a3,NaN,TRBJ1-1*01,NaN,TRBV6-4*01,NaN,CASSDVLMEGLRHTEAFF,RPIIRPATL,HLA-B*08:01,B2M,I,human,InfluenzaA,NP,VDJdb,NaN,NaN
2,0000df14-c075-46f1-83e3-45637a709fd7,TRAJ44*01,TRBJ2-5*01,TRAV4*01,TRBV6-1*01,CLVGGTGTASKLTF,CASRETGGVWETQYF,CINGVCWTV,HLA-A*02:01,B2M,I,human,HCV,NS3,VDJdb,NaN,NaN
3,0000df14-c075-46f1-83e3-45637a709fd7,TRAJ44*01,TRBJ2-5*01,TRAV4*01,TRBV6-1*01,CLVGGTGTASKLTF,CASRETGGVWETQYF,CINGVCWTV,HLA-A*02:01,B2M,I,human,HCV,NS3,VDJdb,NaN,NaN
4,000123c6-40a8-47f6-a9d6-30dbfe0b68ce,NaN,TRBJ2-1*01,NaN,TRBV9*01,NaN,CASNLAGNNEQFF,GLCTLVAML,HLA-A*02:01,B2M,I,human,EBV,BMLF1,VDJdb,NaN,NaN


In [15]:
raw_data.shape

(203031, 17)